In [5]:
!pip install pvlib

In [22]:
import pandas as pd
from pvlib import pvsystem, temperature
import requests
import numpy as np

In [7]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -7.1195,
    "longitude": -34.8450,
    "start_date": "2023-01-01",
    "end_date": "2024-12-31",
    "hourly": "shortwave_radiation,direct_radiation,diffuse_radiation,global_tilted_irradiance,temperature_2m,wind_speed_10m,cloud_cover",
    "tilt": 7,
    "azimuth": 0,
    "timezone": "America/Recife",
}

resposta = requests.get(url, params=params, timeout=60)
resposta.raise_for_status()  # dispara erro se algo deu errado (ex: parâmetro inválido)
dados = resposta.json()

# Os dados horários vêm dentro da chave "hourly"
df = pd.DataFrame(dados["hourly"])
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")

print(df.head())
df.to_csv("dados_era5_joao_pessoa.csv")

                     shortwave_radiation  direct_radiation  diffuse_radiation  \
time                                                                            
2023-01-01 00:00:00                  0.0               0.0                0.0   
2023-01-01 01:00:00                  0.0               0.0                0.0   
2023-01-01 02:00:00                  0.0               0.0                0.0   
2023-01-01 03:00:00                  0.0               0.0                0.0   
2023-01-01 04:00:00                  0.0               0.0                0.0   

                     global_tilted_irradiance  temperature_2m  wind_speed_10m  \
time                                                                            
2023-01-01 00:00:00                       0.0            23.2             5.1   
2023-01-01 01:00:00                       0.0            23.3             6.5   
2023-01-01 02:00:00                       0.0            23.1             5.1   
2023-01-01 03:00:00        

In [8]:
# --- Passo 1: temperatura da célula do painel ---
# O painel não fica na temperatura do ar — ele esquenta com o sol e esfria com o vento.
# Isso importa porque painéis solares perdem eficiência quando ficam muito quentes.
# Usamos o modelo de Faiman (padrão usado pelo software PVsyst), que já vem pronto no pvlib:
temp_celula = temperature.faiman(
    df["global_tilted_irradiance"],
    df["temperature_2m"],
    df["wind_speed_10m"],
)

# --- Passo 2: potência DC gerada, via modelo PVWatts ---
POTENCIA_SISTEMA_W = 5000   # ajuste pro tamanho do sistema que você quer simular
GAMMA_PDC = -0.004          # coeficiente de temperatura típico de painel de silício (-0.4%/°C)

pdc = pvsystem.pvwatts_dc(
    effective_irradiance=df["global_tilted_irradiance"],  # já é a irradiância no plano do painel
    temp_cell=temp_celula,
    pdc0=POTENCIA_SISTEMA_W,
    gamma_pdc=GAMMA_PDC,
)

# --- Passo 3: perdas do sistema (inversor, cabeamento, sujeira nos painéis, etc.) ---
EFICIENCIA_SISTEMA = 0.86  # valor típico de mercado (perdas totais ~14%)

df["geracao_estimada_kwh"] = (pdc / 1000) * EFICIENCIA_SISTEMA  # W -> kWh, base horária

# --- Passo 4: agregações (sua variável target em diferentes granularidades) ---
target_diario = df["geracao_estimada_kwh"].resample("D").sum()
target_mensal = df["geracao_estimada_kwh"].resample("ME").sum()
target_anual = df["geracao_estimada_kwh"].resample("YE").sum()

#LIMPEZA DOS DADOS

In [9]:
df.head()

,shortwave_radiation,direct_radiation,diffuse_radiation,global_tilted_irradiance,temperature_2m,wind_speed_10m,cloud_cover,geracao_estimada_kwh
time,,,,,,,,
2023-01-01 00:00:00,0.0,0.0,0.0,0.0,23.2,5.1,100,0.0
2023-01-01 01:00:00,0.0,0.0,0.0,0.0,23.3,6.5,100,0.0
2023-01-01 02:00:00,0.0,0.0,0.0,0.0,23.1,5.1,100,0.0
2023-01-01 03:00:00,0.0,0.0,0.0,0.0,23.0,5.4,100,0.0
2023-01-01 04:00:00,0.0,0.0,0.0,0.0,23.3,9.4,100,0.0


In [10]:
df.tail()

,shortwave_radiation,direct_radiation,diffuse_radiation,global_tilted_irradiance,temperature_2m,wind_speed_10m,cloud_cover,geracao_estimada_kwh
time,,,,,,,,
2024-12-31 19:00:00,0.0,0.0,0.0,0.0,27.1,15.4,58,0.0
2024-12-31 20:00:00,0.0,0.0,0.0,0.0,26.8,15.5,43,0.0
2024-12-31 21:00:00,0.0,0.0,0.0,0.0,26.8,15.5,57,0.0
2024-12-31 22:00:00,0.0,0.0,0.0,0.0,26.1,14.3,90,0.0
2024-12-31 23:00:00,0.0,0.0,0.0,0.0,25.8,16.2,82,0.0


In [11]:
df.shape

(17544, 8)

In [12]:
df.columns

Index(['shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
       'global_tilted_irradiance', 'temperature_2m', 'wind_speed_10m',
       'cloud_cover', 'geracao_estimada_kwh'],
      dtype='object')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17544 entries, 2023-01-01 00:00:00 to 2024-12-31 23:00:00
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   shortwave_radiation       17544 non-null  float64
 1   direct_radiation          17544 non-null  float64
 2   diffuse_radiation         17544 non-null  float64
 3   global_tilted_irradiance  17544 non-null  float64
 4   temperature_2m            17544 non-null  float64
 5   wind_speed_10m            17544 non-null  float64
 6   cloud_cover               17544 non-null  int64  
 7   geracao_estimada_kwh      17544 non-null  float64
dtypes: float64(7), int64(1)
memory usage: 1.2 MB


In [14]:
df.describe()

,shortwave_radiation,direct_radiation,diffuse_radiation,global_tilted_irradiance,temperature_2m,wind_speed_10m,cloud_cover,geracao_estimada_kwh
count,17544.000000,17544.000000,17544.000000,17544.000000,17544.000000,17544.000000,17544.000000,17544.000000
mean,251.991108,186.292750,65.698358,248.332860,26.385676,15.537198,57.250399,1.034325
std,327.619253,261.943498,83.953735,323.962902,1.918706,5.035434,34.040315,1.341906
min,0.000000,0.000000,0.000000,0.000000,20.400000,0.400000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,25.000000,12.300000,26.000000,0.000000
50%,12.000000,3.000000,8.000000,11.100000,26.300000,15.500000,53.000000,0.047778
75%,524.000000,361.000000,123.000000,513.400000,27.700000,18.800000,96.000000,2.152645
max,1056.000000,951.000000,456.000000,1075.000000,31.700000,31.700000,100.000000,4.393597


In [15]:
df.isnull().sum()

shortwave_radiation         0
direct_radiation            0
diffuse_radiation           0
global_tilted_irradiance    0
temperature_2m              0
wind_speed_10m              0
cloud_cover                 0
geracao_estimada_kwh        0
dtype: int64

Segundo a análise, não haverá necessidade de limpeza. Todos os dados estão limpos, sem Outliers ou valores nulos.

#APLICAÇÃO DOS MODELOS DE MACHINE LEARNING


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [17]:
X = df.drop("geracao_estimada_kwh", axis = 1)
y = df['geracao_estimada_kwh']

In [18]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size= 0.2, random_state= 42)

In [19]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [20]:
y_pred = model.predict(X_test)

In [23]:
# 1. R² Score (Coeficiente de Determinação)
# Diz o quão bem o seu modelo explica a variação dos dados (vai de 0 a 1, quanto maior, melhor)
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2:.4f}")

# 2. MAE (Erro Médio Absoluto)
# Mostra, em média, o quanto as previsões do modelo erram (na mesma unidade dos seus dados)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.4f}")

# 3. RMSE (Raiz do Erro Quadrático Médio)
# Parecido com o MAE, mas penaliza erros maiores de forma mais severa
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.4f}")


R² Score: 0.9999
MAE: 0.0099
RMSE: 0.0140


In [ ]:
import joblib

In [ ]:
joblib.dump(model, 'modelo_geracao_solar.pkl')

In [ ]:
colunas_features = list(X_train.columns)  # ajuste pro nome real da sua variável de features
joblib.dump(colunas_features, "colunas_features.pkl")